In [1]:
import jpype
import jpype.imports
from jpype.types import *
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from functools import partial

In [2]:
# =========================================
# 1. Data Loader
# =========================================
def load_graphs_from_txt(path: str) -> list:
    graphs = []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows = [list(map(float, row.split(","))) for row in line.split(":")]
            graphs.append(np.array(rows))
    return graphs

def load_labels(path: str):
    with open(path, "r") as f:
        return np.array([line.strip() for line in f if line.strip()])

In [3]:
# =========================================
# 2. Java Bridge Setup
# =========================================
def start_jvm_if_needed():
    if not jpype.isJVMStarted():
        classpath = "."
        jpype.startJVM(classpath=[classpath])

def to_java_object_array(matrix: np.ndarray):
    rows = []
    for i in range(matrix.shape[0]):
        row = [JDouble(float(x)) for x in matrix[i]]
        rows.append(row)
    return jpype.JArray(jpype.JArray(JDouble))(rows)


In [4]:
# =========================================
# 3. Distance Callable Wrapper
# =========================================
def make_java_distance_fn(wl, graphs):
    """Return a callable for sklearn that computes WLDistance2 between graphs[i] and graphs[j]."""
    def distance_fn(i_index, j_index):
        gA = graphs[int(i_index[0])]  # sklearn passes [index] arrays
        gB = graphs[int(j_index[0])]
        java_a = to_java_object_array(gA)
        java_b = to_java_object_array(gB)
        return float(wl.compute(java_a, java_b))
    return distance_fn

In [6]:
# =========================================
# 4. Main
# =========================================
if __name__ == "__main__":
    start_jvm_if_needed()
    DistanceClass = jpype.JClass("distance.graph.WLDistance2")
    wl = DistanceClass()

    train_graphs = load_graphs_from_txt("Data/proteins_train_data2.txt")
    val_graphs = load_graphs_from_txt("Data/proteins_val_data2.txt")
    train_labels = load_labels("Data/proteins_train_labels2.txt")
    val_labels = load_labels("Data/proteins_val_labels2.txt")
    test_graphs = load_graphs_from_txt("Data/proteins_test_data2.txt")
    test_labels = load_labels("Data/proteins_test_labels2.txt")

    print(f"Loaded {len(train_graphs)} train and {len(val_graphs)} validation and {len(test_graphs)} graphs")

    # Build a wrapper around the training graphs for sklearn
    distance_fn = make_java_distance_fn(wl, train_graphs)

    # sklearn requires numerical feature arrays — we’ll pass indices instead
    X_train = np.arange(len(train_graphs)).reshape(-1, 1)
    X_val = np.arange(len(val_graphs)).reshape(-1, 1)
    X_test = np.arange(len(test_graphs)).reshape(-1, 1)

    # Create KNN with callable metric
    knn = KNeighborsClassifier(n_neighbors=5, metric=distance_fn, n_jobs=-1)
    knn.fit(X_train, train_labels)

    preds = knn.predict(X_val)
    acc = np.mean(preds == val_labels)
    preds2 = knn.predict(X_test)
    acc2 = np.mean(preds2 == test_labels)
    print(f"\nValidation accuracy: {acc:.4f}")
    print(f"\nTest accuracy: {acc2:.4f}")

    jpype.shutdownJVM()

Loaded 890 train and 111 validation and 112 graphs

Validation accuracy: 0.5405

Test accuracy: 0.6250
